## Create Mask

Use this notebook to draw exclusion masks on wells using Napari.
The **well selector** shows all wells from the active experiment config.
Wells with completed masks appear in green (✓).

**Workflow:**
1. Run the config cell — paths load automatically from the active experiment
2. Run the selector cell and click a well
3. Run the load cell to open that well's images
4. Run the Napari cell, draw the mask, click **Save Mask** in Napari
5. Run the mark-done cell to record the well as complete


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from project_config import load_config

config     = load_config()
base_dir   = config['base_dir']
well_list  = config['well_list']

path_df    = os.path.join(base_dir, 'output_df')
path_tiffs = os.path.join(base_dir, 'aligned_tiffs')
path_save  = os.path.join(base_dir, 'masks')
use_existing = True

exp_label = config.get('experiment_name', config.get('experiment_key', '?'))
print(f'Experiment: {exp_label}')
print(f'Base dir:   {base_dir}')
print(f'Wells:      {well_list}')

In [ ]:
from well_plate_selector import SingleWellSelector
from mask_status import get_masked_wells

masked   = get_masked_wells(base_dir)
selector = SingleWellSelector(selectable_wells=well_list, masked_wells=masked)
selector.show()

**After clicking a well above**, run the cell below to load that well's images.

In [ ]:
import pickle
import pandas as pd
from tifffile import imread

selected_well = selector.well
if selected_well is None:
    raise RuntimeError('No well selected — click a well on the plate above first.')

print(f'Loading: {selected_well}')
os.makedirs(path_save, exist_ok=True)

df_name = f'df_{selected_well}.pkl'
with open(os.path.join(path_df, df_name), 'rb') as fh:
    df = pickle.load(fh)

im_path = os.path.join(path_tiffs, selected_well, df.aligned_file_name[0])
im = imread(im_path)
print(f'Image shape: {im.shape}')

mask_exist = False
if use_existing:
    try:
        mask_path = os.path.join(path_save, f'mask_{selected_well}.pkl')
        mask = pickle.load(open(mask_path, 'rb'))
        print('Existing mask found.')
        mask_exist = True
    except FileNotFoundError:
        print('No existing mask — starting fresh.')

## Open Napari to draw the mask

Draw exclusion regions on the **mask** layer, then click **Save Mask** in the Napari panel.

In [ ]:
import napari
from magicgui.widgets import Dropdown
from magicgui import magicgui

channel_list = ['_'.join([str(x).zfill(2), y]) for x, y in zip(df.nameRound, df.signal.to_list())]
df['round_channel'] = channel_list

def save_mask(viewer: napari.Viewer):
    mask = viewer.layers['mask'].data
    mask_path = os.path.join(path_save, f'mask_{selected_well}.pkl')
    pickle.dump(mask, open(mask_path, 'wb'))
    viewer.status = 'Mask saved.'

save_data = magicgui(save_mask, call_button='Save Mask')

def channel_change(val_coming: str):
    global viewer
    alig_file = df.loc[df.round_channel == val_coming, 'aligned_file_name'].to_list()[0]
    im_new = imread(os.path.join(path_tiffs, selected_well, alig_file))
    viewer.layers[1].data = im_new
    viewer.layers[1].name = df.loc[df.round_channel == val_coming, 'signal'].to_list()[0]
    viewer.status = 'Channel changed.'

widget_channel = Dropdown(choices=channel_list)
widget_channel.changed.connect(channel_change)

In [ ]:
viewer = napari.Viewer()
viewer.add_shapes(name='mask', face_color='red')
if use_existing and mask_exist:
    viewer.layers['mask'].data = mask
viewer.add_image(im, blending='additive', contrast_limits=[0, 4095], name=df.signal[0])
viewer.window.add_dock_widget(widget_channel, area='left', name='Choose channel')
viewer.window.add_dock_widget(save_data, area='left', name='Save Mask')

## Mark well as complete

After saving the mask in Napari, run this cell to record the well as done.
It will appear grayed out (✓) next time you run the selector cell.

In [ ]:
from mask_status import mark_well_masked
mark_well_masked(selected_well, base_dir)
print(f'✓ Mask recorded for {selected_well}.')
print('Re-run the selector cell (mask_selector) to see the updated plate.')